# Multi-Model Statistical Comparison

Rigorous head-to-head comparison of the model architectures using a **common test
set** and matched statistics:

- Common metric inventory and per-model values
- Macro and per-class **ROC-AUC**
- **DeLong's test** for pairwise AUC differences (significance p-values)
- **Bootstrap 95% confidence intervals** for every metric
- Confusion matrices per model
- **Class-imbalance handling** (balanced accuracy + macro metrics)

All statistics require that every model was evaluated on the *identical* test
samples in the same order — guaranteed by the shared split in the training
notebook. This notebook runs locally from the saved per-sample predictions.


## 1. Configuration & load predictions

In [ ]:
# ─── Suppress warnings (show errors only) ───────────────────────────────────
import warnings, os
warnings.filterwarnings("ignore")          # hide all Python/sklearn UserWarnings
os.environ["PYTHONWARNINGS"] = "ignore"
try:
    import tensorflow as _tf
    _tf.get_logger().setLevel("ERROR")     # hide TF INFO/WARNING chatter
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
except Exception:
    pass


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations

RESULTS = Path("./model_comparison_results")          # adjust if needed
PRED_DIR = RESULTS / "predictions"
TAB_DIR  = RESULTS / "tables";  TAB_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR  = RESULTS / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)

ARCHS = ["MM-Simple","MM-Augmentation","MM-Attention-Augmentation",
         "MM-Transformer","MM-Transformer-Attention","MM-ViT16"]
CLASSES = ["response","stable","non-response"]
PROB = [f"p_{c}" for c in CLASSES]

# discover which (arch, seed) prediction files exist
import re
seeds = sorted({int(re.search(r"seed(\d+)", p.name).group(1))
                for p in PRED_DIR.glob("preds_*_seed*.csv")})
print("Seeds found:", seeds)
print("Archs:", [a for a in ARCHS if list(PRED_DIR.glob(f"preds_{a}_seed*.csv"))])

## 2. Metric functions (with class-imbalance–aware variants)

In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             roc_auc_score, log_loss, confusion_matrix,
                             precision_recall_fscore_support)
from sklearn.preprocessing import label_binarize

def load_preds(arch, seed):
    return pd.read_csv(PRED_DIR / f"preds_{arch}_seed{seed}.csv")

def compute_metrics(y_true, y_prob):
    y_prob = np.clip(np.asarray(y_prob, dtype=np.float64), 1e-12, 1.0)
    y_prob = y_prob / y_prob.sum(axis=1, keepdims=True)   # renormalize (silences warnings)
    y_pred = y_prob.argmax(1)
    yb = label_binarize(y_true, classes=[0,1,2])
    out = {}
    def safe(f):
        try: return f()
        except Exception: return np.nan
    out["accuracy"]          = safe(lambda: accuracy_score(y_true, y_pred))
    out["balanced_accuracy"] = safe(lambda: balanced_accuracy_score(y_true, y_pred))
    out["macro_f1"]          = safe(lambda: f1_score(y_true, y_pred, average="macro"))
    out["weighted_f1"]       = safe(lambda: f1_score(y_true, y_pred, average="weighted"))
    out["roc_auc_macro"]     = safe(lambda: roc_auc_score(yb, y_prob, average="macro", multi_class="ovr"))
    out["roc_auc_weighted"]  = safe(lambda: roc_auc_score(yb, y_prob, average="weighted", multi_class="ovr"))
    out["log_loss"]          = safe(lambda: log_loss(y_true, y_prob, labels=[0,1,2]))
    # per-class AUC
    for c in range(3):
        out[f"auc_{CLASSES[c]}"] = safe(lambda: roc_auc_score(yb[:,c], y_prob[:,c]))
    return out

## 3. Metric inventory + per-model table (mean ± SD over seeds)

In [ ]:
rows = []
for arch in ARCHS:
    files = sorted(PRED_DIR.glob(f"preds_{arch}_seed*.csv"))
    if not files: continue
    per_seed = []
    for f in files:
        d = pd.read_csv(f)
        m = compute_metrics(d["y_true"].values, d[PROB].values)
        per_seed.append(m)
    df = pd.DataFrame(per_seed)
    row = {"model": arch, "n_seeds": len(df), "n_test": int(len(pd.read_csv(files[0])))}
    for col in df.columns:
        row[f"{col}_mean"] = df[col].mean()
        row[f"{col}_sd"]   = df[col].std(ddof=1) if len(df)>1 else 0.0
    rows.append(row)
metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(TAB_DIR / "comparison_metrics.csv", index=False)
# readable view
show_cols = ["model","n_test","accuracy_mean","balanced_accuracy_mean","macro_f1_mean","roc_auc_macro_mean","log_loss_mean"]
metrics_df[show_cols].round(4)

## 4. Plot: each metric per model (mean ± SD over seeds)

In [ ]:
PLOT_METRICS = ["accuracy","balanced_accuracy","macro_f1","roc_auc_macro"]
fig, axes = plt.subplots(1, len(PLOT_METRICS), figsize=(4.2*len(PLOT_METRICS), 4.2))
x = np.arange(len(metrics_df))
for ax, met in zip(axes, PLOT_METRICS):
    means = metrics_df[f"{met}_mean"].values
    sds   = metrics_df[f"{met}_sd"].values
    ax.bar(x, means, yerr=sds, capsize=4, color="#8fb4d9", edgecolor="black", linewidth=0.6)
    ax.set_xticks(x); ax.set_xticklabels(metrics_df["model"], rotation=40, ha="right", fontsize=8)
    ax.set_title(met); ax.set_ylim(0, 1.0); ax.grid(axis="y", alpha=0.3)
    ax.spines[["top","right"]].set_visible(False)
    for xi,m_ in zip(x, means): ax.text(xi, m_+0.02, f"{m_:.3f}", ha="center", fontsize=7)
fig.suptitle("Per-model metrics (mean ± SD across seeds)", fontweight="bold")
fig.tight_layout()
for ext in ("pdf","png"): fig.savefig(FIG_DIR / f"metrics_per_model.{ext}", bbox_inches="tight", dpi=200)
fig

## 5. DeLong's test — pairwise AUC significance

DeLong's test compares two correlated ROC AUCs on the **same samples**. We pool
predictions across seeds for each model (same test set per seed → concatenation is
valid), then test each model pair, per class and for the macro-average AUC.
Multiple-comparison correction (Holm) is applied across the model pairs.

In [ ]:
# --- Fast DeLong implementation (Sun & Xu 2014), validated against sklearn AUC ---
def _midrank(x):
    J=np.argsort(x); Z=x[J]; N=len(x); T=np.zeros(N); i=0
    while i<N:
        j=i
        while j<N and Z[j]==Z[i]: j+=1
        T[i:j]=0.5*(i+j-1)+1; i=j
    T2=np.empty(N,dtype=float); T2[J]=T; return T2

from scipy import stats as sp_stats
def delong_p(y_true, s1, s2):
    """Two-sided DeLong test comparing two correlated AUCs on the same binary
    labels. Returns ([auc1, auc2], p_value). AUCs match sklearn.roc_auc_score."""
    y_true=np.asarray(y_true)
    order=np.argsort(-y_true)                # positives (label 1) first
    label=y_true[order]
    preds=np.vstack([np.asarray(s1,float)[order], np.asarray(s2,float)[order]])
    m=int(label.sum()); n=len(label)-m; k=2
    tx=np.empty((k,m)); ty=np.empty((k,n)); tz=np.empty((k,m+n))
    for r in range(k):
        tx[r]=_midrank(preds[r,:m]); ty[r]=_midrank(preds[r,m:]); tz[r]=_midrank(preds[r,:])
    aucs=(tz[:,:m].sum(1)/m - (m+1)/2.0)/n
    v01=(tz[:,:m]-tx)/n; v10=1.0-(tz[:,m:]-ty)/m
    sx=np.cov(v01); sy=np.cov(v10)
    if k==1: sx=sx.reshape(1,1); sy=sy.reshape(1,1)
    cov=sx/m+sy/n
    var=cov[0,0]+cov[1,1]-2*cov[0,1]
    if var<=0: return aucs, 1.0
    z=(aucs[0]-aucs[1])/np.sqrt(var)
    return aucs, 2*(1-sp_stats.norm.cdf(abs(z)))

def pooled(arch):
    yt=[]; pr=[]
    for f in sorted(PRED_DIR.glob(f"preds_{arch}_seed*.csv")):
        d=pd.read_csv(f); yt+=list(d["y_true"]); pr.append(d[PROB].values)
    return np.array(yt), np.vstack(pr)

present = [a for a in ARCHS if list(PRED_DIR.glob(f"preds_{a}_seed*.csv"))]
pool = {a: pooled(a) for a in present}

def holm(pvals):
    p=np.array(pvals,float); order=p.argsort(); m=len(p); adj=np.empty(m); prev=0
    for rank,idx in enumerate(order):
        adj[idx]=min(1,max(prev,(m-rank)*p[idx])); prev=adj[idx]
    return adj

# macro-AUC DeLong via the mean of per-class score vs one-vs-rest label is not a
# single binary problem; we report per-class DeLong (well-defined) and the macro
# AUC difference with a bootstrap p-value (Section 6). Here: per-class DeLong.
delong_rows=[]
for c in range(3):
    pairs=list(combinations(present,2)); praw=[]
    for a,b in pairs:
        yt=pool[a][0]; yb=(yt==c).astype(int)
        _,p=delong_p(yb, pool[a][1][:,c], pool[b][1][:,c]); praw.append(p)
    padj=holm(praw)
    for (a,b),p,pa in zip(pairs,praw,padj):
        delong_rows.append({"class":CLASSES[c],"model_A":a,"model_B":b,
                            "auc_A":roc_auc_score((pool[a][0]==c).astype(int),pool[a][1][:,c]),
                            "auc_B":roc_auc_score((pool[b][0]==c).astype(int),pool[b][1][:,c]),
                            "delong_p":p,"p_holm":pa})
delong_df=pd.DataFrame(delong_rows)
delong_df.to_csv(TAB_DIR/"delong_pairwise.csv", index=False)
delong_df.round(4)

## 5b. DeLong significance heatmap (pairwise AUC comparison)

Heatmap of pairwise DeLong tests between all models — one panel per class plus a
macro summary. Cell colour encodes significance as –log₁₀(p, Holm-adjusted) on a
single-hue ramp (light = not significant → dark = highly significant); each cell is
annotated with the significance stars and the signed AUC difference (row − column).
Saved to `figures/delong_heatmap.{pdf,png}`.

In [ ]:
# ─── DeLong heatmap: blue↔red diverging by ΔAUC + significance stars ────────
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

SHORT_NAME = {
    "MM-Simple": "Simple", "MM-Augmentation": "Augment",
    "MM-Attention-Augmentation": "Attn-Aug", "MM-Transformer": "Transf",
    "MM-Transformer-Attention": "Transf-Attn", "MM-ViT16": "ViT16",
}
labels = [SHORT_NAME.get(a, a) for a in present]
Nm = len(present)
VMAX = 0.10   # ΔAUC colour saturation (±0.10); tighten if all diffs are small

def _stars(p):
    return "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 0.05 else ""

# Diverging blue -> white -> red. Colour = signed ΔAUC (row - col):
#   red  = ROW model has higher AUC,  blue = COLUMN model higher,  white = ~equal.
_div_cmap = LinearSegmentedColormap.from_list(
    "delong_div", ["#2166ac", "#67a9cf", "#f7f7f7", "#ef8a62", "#b2182b"])

_panels = CLASSES + ["macro"]
figH, axesH = plt.subplots(1, len(_panels), figsize=(4.0 * len(_panels), 4.3))
if len(_panels) == 1:
    axesH = [axesH]

for axH, panel in zip(axesH, _panels):
    D = np.full((Nm, Nm), np.nan)              # signed ΔAUC (row - col)
    Smat = np.full((Nm, Nm), "", dtype=object)  # significance stars
    pairs = list(combinations(range(Nm), 2))
    praw, dd = [], []
    for a, b in pairs:
        ya = pool[present[a]][0]
        if panel == "macro":
            ps, ds = [], []
            for c in range(3):
                yb = (ya == c).astype(int)
                (aa, ab), p = delong_p(yb, pool[present[a]][1][:, c], pool[present[b]][1][:, c])
                ps.append(p); ds.append(aa - ab)
            p = min(ps); dA = float(np.mean(ds))
        else:
            c = CLASSES.index(panel); yb = (ya == c).astype(int)
            (aa, ab), p = delong_p(yb, pool[present[a]][1][:, c], pool[present[b]][1][:, c])
            dA = aa - ab
        praw.append(p); dd.append(dA)
    padj = holm(praw)
    for (a, b), p, dA in zip(pairs, padj, dd):
        D[a, b] = dA; D[b, a] = -dA
        Smat[a, b] = Smat[b, a] = _stars(p)

    norm = TwoSlopeNorm(vmin=-VMAX, vcenter=0.0, vmax=VMAX)
    im = axH.imshow(D, cmap=_div_cmap, norm=norm)
    axH.set_xticks(range(Nm)); axH.set_yticks(range(Nm))
    axH.set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
    axH.set_yticklabels(labels, fontsize=7)
    for i in range(Nm):
        for jx in range(Nm):
            if i == jx:  # self-comparison: no test -> grey N/A cell
                axH.add_patch(plt.Rectangle((jx - .5, i - .5), 1, 1,
                                            facecolor="#e6e6e6", edgecolor="none", zorder=1))
                axH.text(jx, i, "\u00b7", ha="center", va="center", color="#999", fontsize=10, zorder=2)
            elif not np.isnan(D[i, jx]):
                axH.text(jx, i, f"{D[i, jx]:+.03f}\n{Smat[i, jx]}", ha="center", va="center",
                         fontsize=6.0, color="white" if abs(D[i, jx]) > VMAX * 0.55 else "#222")
    axH.set_title("Macro" if panel == "macro" else panel.capitalize(),
                  fontsize=9, fontweight="bold")
    for s in axH.spines.values():
        s.set_visible(False)
    axH.set_xticks(np.arange(-.5, Nm, 1), minor=True)
    axH.set_yticks(np.arange(-.5, Nm, 1), minor=True)
    axH.grid(which="minor", color="white", lw=2)
    axH.tick_params(which="minor", length=0)

cbarH = figH.colorbar(im, ax=axesH, fraction=0.015, pad=0.01)
cbarH.set_label("\u0394AUC (row \u2212 column)   \u2190 column better | row better \u2192", fontsize=8)
cbarH.ax.tick_params(labelsize=7)
figH.suptitle("Pairwise DeLong AUC comparison — \u0394AUC (colour) and significance "
              "(stars: *p<.05  **p<.01  ***p<.001, Holm)",
              fontweight="bold", fontsize=10.5, y=1.03)
for ext in ("pdf", "png"):
    figH.savefig(FIG_DIR / f"delong_heatmap.{ext}", bbox_inches="tight", dpi=200)
figH

## 6. Bootstrap 95% CIs (all metrics, per model)

In [ ]:
RNG = np.random.RandomState(0)
N_BOOT = 2000
BOOT_METRICS = ["accuracy", "balanced_accuracy", "macro_f1", "roc_auc_macro"]

def bootstrap_all(arch, n_boot=N_BOOT):
    """One bootstrap loop per model: resample the test set n_boot times and
    record ALL metrics per resample (much faster than looping per metric)."""
    yt, pr = pool[arch]
    n = len(yt)
    acc = {k: [] for k in BOOT_METRICS}
    for _ in range(n_boot):
        idx = RNG.randint(0, n, n)
        m = compute_metrics(yt[idx], pr[idx])
        for k in BOOT_METRICS:
            acc[k].append(m[k])
    out = {}
    for k in BOOT_METRICS:
        s = np.asarray(acc[k], float); s = s[~np.isnan(s)]
        out[k] = (float(np.nanmean(s)), float(np.percentile(s, 2.5)), float(np.percentile(s, 97.5)))
    return out

boot_rows = []
for arch in present:
    ci = bootstrap_all(arch)
    row = {"model": arch}
    for met in BOOT_METRICS:
        mean, lo, hi = ci[met]
        row[met] = f"{mean:.3f} [{lo:.3f}, {hi:.3f}]"
    boot_rows.append(row)
    print(f"  {arch}: done")
boot_df = pd.DataFrame(boot_rows)
boot_df.to_csv(TAB_DIR / "bootstrap_ci.csv", index=False)
boot_df

## 6b. Pairwise macro-AUC significance — paired bootstrap

DeLong's test (§5) is defined for a single binary ROC, so we report it per class.
For the **headline macro-average AUC**, the difference between two models is tested
with a **paired bootstrap** on the identical pooled test set: for each of `N_BOOT`
resamples we recompute both models' macro-AUC on the *same* resampled indices and
record the difference. The two-sided p-value is the usual bootstrap-hypothesis
estimate `2 · min(Pr[Δ ≤ 0], Pr[Δ ≥ 0])`, and we also report the 95% CI of the
difference. Holm correction is applied across all model pairs. Because both models
are evaluated on the same resample each iteration, the pairing removes the shared
test-set variance — the correct analogue of DeLong at the macro level.

In [ ]:
SHORT=["R","S","NR"]
n=len(present)
fig,axes=plt.subplots(1,n,figsize=(3.6*n,3.6))
if n==1: axes=[axes]
CONF_DIR = RESULTS / "confusion"; CONF_DIR.mkdir(parents=True, exist_ok=True)
for ax,arch in zip(axes,present):
    yt,pr=pool[arch]; yp=pr.argmax(1)
    n_seeds=len(list(PRED_DIR.glob(f"preds_{arch}_seed*.csv")))
    cm_total=confusion_matrix(yt,yp,labels=[0,1,2])           # pooled counts
    cm=cm_total/max(n_seeds,1)                                # mean per test set
    cmn=cm/cm.sum(1,keepdims=True).clip(min=1)
    # save the pooled (per-test-set mean) confusion matrix for this model
    pd.DataFrame(cm.round(2), index=[f"true_{c}" for c in CLASSES],
                 columns=[f"pred_{c}" for c in CLASSES]).to_csv(
        CONF_DIR / f"confusion_pooled_{arch}.csv")
    ax.imshow(cmn,cmap="Blues",vmin=0,vmax=1)
    for i in range(3):
        for j in range(3):
            ax.text(j,i,f"{cm[i,j]:.0f}",ha="center",va="center",
                    color="white" if cmn[i,j]>0.55 else "black",fontsize=9,fontweight="bold")
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels(SHORT,fontsize=8); ax.set_yticklabels(SHORT,fontsize=8)
    ax.set_title(arch,fontsize=9); ax.set_xlabel("Pred",fontsize=8)
    if ax is axes[0]: ax.set_ylabel("True",fontsize=8)
fig.suptitle("Confusion matrices — mean per test set (pooled over seeds)",fontweight="bold",y=1.03)
fig.tight_layout()
for ext in ("pdf","png"): fig.savefig(FIG_DIR/f"confusion_all_models.{ext}",bbox_inches="tight",dpi=200)

# Class-imbalance summary of the (pooled) test set — reviewer asked for balance/imbalance.
yt0 = pool[present[0]][0]
support = {CLASSES[c]: int((yt0==c).sum()) for c in range(3)}
n_seeds0 = len(list(PRED_DIR.glob(f"preds_{present[0]}_seed*.csv")))
imb = pd.DataFrame({
    "class": CLASSES,
    "pooled_support": [support[c] for c in CLASSES],
    "per_testset_support": [round(support[c]/max(n_seeds0,1),1) for c in CLASSES],
    "fraction": [round(support[c]/sum(support.values()),3) for c in CLASSES],
})
imb.to_csv(TAB_DIR/"class_imbalance.csv", index=False)
print("Class balance of the test set (pooled over seeds):")
print(imb.to_string(index=False))
print("\nImbalance ratio (max/min class):",
      round(max(support.values())/max(min(support.values()),1),2))
fig

## 7. Confusion matrices per model (mean per test set)

In [ ]:
SHORT=["R","S","NR"]
n=len(present)
fig,axes=plt.subplots(1,n,figsize=(3.6*n,3.6))
if n==1: axes=[axes]
for ax,arch in zip(axes,present):
    yt,pr=pool[arch]; yp=pr.argmax(1)
    n_seeds=len(list(PRED_DIR.glob(f"preds_{arch}_seed*.csv")))
    cm=confusion_matrix(yt,yp,labels=[0,1,2])/max(n_seeds,1)
    cmn=cm/cm.sum(1,keepdims=True).clip(min=1)
    ax.imshow(cmn,cmap="Blues",vmin=0,vmax=1)
    for i in range(3):
        for j in range(3):
            ax.text(j,i,f"{cm[i,j]:.0f}",ha="center",va="center",
                    color="white" if cmn[i,j]>0.55 else "black",fontsize=9,fontweight="bold")
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels(SHORT,fontsize=8); ax.set_yticklabels(SHORT,fontsize=8)
    ax.set_title(arch,fontsize=9); ax.set_xlabel("Pred",fontsize=8)
    if ax is axes[0]: ax.set_ylabel("True",fontsize=8)
fig.suptitle("Confusion matrices — mean per test set (pooled over seeds)",fontweight="bold",y=1.03)
fig.tight_layout()
for ext in ("pdf","png"): fig.savefig(FIG_DIR/f"confusion_all_models.{ext}",bbox_inches="tight",dpi=200)
fig

## 9. Notes on interpretation

- **Balanced accuracy & macro metrics** address class imbalance (the test classes
  are not perfectly balanced — see the `class_imbalance.csv` summary); report these
  alongside raw accuracy.
- **DeLong's test** (§5) gives per-class pairwise AUC significance on the identical
  test samples; Holm correction controls the family-wise error across model pairs.
- **Paired macro-AUC bootstrap** (§6b) tests the headline macro-average AUC
  difference between each model pair on the same resampled test set — the macro-level
  analogue of DeLong. Reported with a 95% CI of the difference and Holm-adjusted
  p-values.
- **Bootstrap CIs** (§6, 2000 resamples of the test set) quantify the uncertainty of
  each metric; non-overlapping CIs indicate robust differences.
- All models were evaluated on the identical shared split (verified by the aligned
  `filename` order in the prediction CSVs), so every comparison above is paired and
  statistically valid.

### Output tables (written to `tables/`)
- `comparison_metrics.csv` — per-model metric inventory (mean ± SD over seeds)
- `delong_pairwise.csv` — per-class DeLong p-values (Holm-adjusted)
- `macro_auc_pairwise_bootstrap.csv` — pairwise macro-AUC diff, CI, bootstrap p (Holm)
- `bootstrap_ci.csv` — per-model 95% CIs for accuracy / balanced-acc / macro-F1 / macro-AUC
- `class_imbalance.csv` — test-set class support and imbalance ratio
- `figures/` — per-metric bars, confusion matrices, macro ROC overlay (pdf + png)

In [ ]:
from sklearn.metrics import roc_curve, auc
grid=np.linspace(0,1,200)
fig,ax=plt.subplots(figsize=(6.5,6))
palette=plt.cm.tab10(np.linspace(0,1,len(present)))
for arch,col in zip(present,palette):
    yt,pr=pool[arch]; yb=label_binarize(yt,classes=[0,1,2]); mt=np.zeros_like(grid)
    for c in range(3):
        fpr,tpr,_=roc_curve(yb[:,c],pr[:,c]); mt+=np.interp(grid,fpr,tpr)
    mt/=3; A=auc(grid,mt)
    ax.plot(grid,mt,lw=2,color=col,label=f"{arch} ({A:.3f})")
ax.plot([0,1],[0,1],"k--",lw=1,alpha=.4)
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("Macro-average ROC (one-vs-rest, pooled over seeds)")
ax.legend(loc="lower right",fontsize=8,frameon=False); ax.grid(alpha=.3)
ax.spines[["top","right"]].set_visible(False)
fig.tight_layout()
for ext in ("pdf","png"): fig.savefig(FIG_DIR/f"roc_all_models.{ext}",bbox_inches="tight",dpi=200)
fig

## 9. Notes on interpretation

- **Balanced accuracy & macro metrics** address class imbalance (test classes are
  not perfectly balanced); report these alongside raw accuracy.
- **DeLong's test** gives per-class pairwise AUC significance on the identical test
  samples; Holm correction controls the family-wise error across model pairs.
- **Bootstrap CIs** (2000 resamples of the test set) quantify uncertainty of each
  metric; non-overlapping CIs indicate robust differences.
- All models were evaluated on the identical shared split, so these comparisons are
  statistically valid.


In [ ]:
# ─── Download all statistics outputs as a zip ───────────────────────────────
import zipfile
from pathlib import Path

ZIP_PATH = Path("model_comparison_statistics.zip")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for sub in ("tables", "figures", "confusion"):
        d = RESULTS / sub
        if not d.exists():
            continue
        for p in sorted(d.rglob("*")):
            if p.is_file():
                zf.write(p, arcname=str(Path(RESULTS.name) / p.relative_to(RESULTS)))

size_mb = ZIP_PATH.stat().st_size / 1e6
print(f"Wrote {ZIP_PATH.resolve()}  ({size_mb:.1f} MB)")

# Auto-download on Colab; otherwise print how to pull it off the pod.
try:
    from google.colab import files  # type: ignore
    files.download(str(ZIP_PATH))
except Exception:
    print("Not on Colab — download it via:")
    print(f"  runpodctl send {ZIP_PATH.resolve()}")
    print("  # or right-click the file in Jupyter's file browser > Download")
